# 2026-IoT 코랩 실행 노트북

위에서부터 차례로 실행한다. 런타임이 끊기면 1번부터 다시 실행한다. 5번 실험 셀은 끝난 실행을 건너뛴다.

**처음 한 번만 할 일**
1. 메뉴 `런타임 > 런타임 유형 변경`에서 GPU(T4)를 고른다.
2. 왼쪽 열쇠 아이콘(Secrets)에 `GH_TOKEN`을 추가하고 이 노트북의 액세스를 켠다.
   - 값은 `combile/2026-IoT` 읽기 권한이 있는 GitHub 토큰이다. 저장소가 비공개라 토큰이 필요하다.

## 0. 설정

In [ ]:
BRANCH = 'main'                  # 아직 합치지 않은 브랜치를 시험할 때는 'feature/defense' 등으로 바꾼다
RESULTS = '/content/drive/MyDrive/2026-IoT/results'   # 결과 CSV와 로그를 드라이브에 남긴다
DATA = '/content/data'           # CIFAR-10은 처음 실행할 때 자동으로 내려받는다

## 1. GPU 확인, 드라이브 연결, 코드 받기

In [ ]:
!nvidia-smi -L
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, subprocess, shutil, pathlib
os.makedirs(RESULTS, exist_ok=True)
token = userdata.get('GH_TOKEN')
shutil.rmtree('/content/FedACG', ignore_errors=True)
shutil.rmtree('/content/2026-IoT', ignore_errors=True)
subprocess.run(['git', 'clone', '-q', 'https://github.com/geehokim/FedACG.git', '/content/FedACG'], check=True)
subprocess.run(['git', 'clone', '-q', '-b', BRANCH, f'https://{token}@github.com/combile/2026-IoT.git',
                '/content/2026-IoT'], check=True)   # 토큰이 화면에 찍히지 않게 subprocess로 실행한다
print(subprocess.run(['git', '-C', '/content/2026-IoT', 'log', '--oneline', '-3'],
                     capture_output=True, text=True).stdout)

## 2. 팀 코드 붙이기

`attacks/`, `metrics/`, `defenses/`와 설정 폴더를 FedACG에 복사하고 패치를 순서대로 적용한다. 아직 저장소에 없는 모듈은 건너뛴다.

In [ ]:
T = pathlib.Path('/content/2026-IoT')
os.chdir('/content/FedACG')
for pkg in ['attacks', 'metrics', 'defenses']:
    if (T / pkg).exists():
        shutil.copytree(T / pkg, pkg, dirs_exist_ok=True)
for grp in (T / 'configs').iterdir():
    if grp.is_dir() and grp.name != 'experiments':
        shutil.copytree(grp, f'configs/{grp.name}', dirs_exist_ok=True)
for p in ['fedacg_integration.patch', 'metrics_integration.patch', 'defense_integration.patch']:
    if (T / 'integration' / p).exists():
        subprocess.run(['git', 'apply', str(T / 'integration' / p)], check=True)
        print('applied', p)
HAS_METRICS = (T / 'metrics').exists()
print('metrics module:', HAS_METRICS)

## 3. 패키지 설치와 테스트

In [ ]:
!pip -q install hydra-core==1.3.2 hydra-colorlog omegaconf imageio coloredlogs tabulate torchinfo torchmetrics wandb pytest
!cd /content/2026-IoT && python -m pytest -q

## 4. 2라운드 점검

모델 오염 30% + Krum으로 2라운드만 돌린다. 오류 없이 끝나는지, 라운드당 몇 초 걸리는지 확인한다. 이 시간으로 전체 실험 시간을 계산한다(라운드당 시간 × 라운드 수 × 실행 횟수).

In [ ]:
COMMON = f'dataset.path={DATA} dataset.client_path=/content/FedACG/client_data wandb=False'
if HAS_METRICS:
    COMMON += f' metrics.results_dir={RESULTS}'
import time
t0 = time.time()
!python federated_train.py server=FedACG client=ACG dataset=cifar10 attack=mp attack.ratio=0.3 attack.start_round=0 attack.end_round=2 +defense=default +server.aggregator=krum trainer.global_rounds=2 trainer.num_clients=100 trainer.participation_rate=0.1 exp_name=smoke {COMMON}
print(f'2라운드 {time.time() - t0:.0f}초 (데이터 다운로드 시간 포함)')

## 5. 실험 실행 (우은식 담당: 모델 오염 90회)

- `SPEC`의 조합(2 알고리즘 × 5 집계 × 3 비율 × 3 시드)을 차례로 돌린다.
- 결과 CSV의 마지막 라운드가 `ROUNDS - 1`이면 끝난 실행으로 보고 건너뛴다. 런타임이 끊기면 1~3번을 다시 실행한 뒤 이 셀을 실행하면 남은 실행부터 이어서 돈다.
- 중간에 끊긴 실행은 처음부터 다시 돈다.
- `ROUNDS`는 첫 회의에서 정할 값이다. 4번에서 잰 라운드당 시간으로 전체 시간을 먼저 계산한다.

In [ ]:
SPEC = '/content/2026-IoT/configs/experiments/mp_sweep.yaml'   # 라벨 플리핑은 lf_sweep.yaml
ROUNDS = 1000                                                   # [TBD] 첫 회의에서 확정

import csv, sys, yaml
sys.path.insert(0, '/content/2026-IoT')
from scripts.make_sweep import iter_runs

def finished(name):
    path = pathlib.Path(RESULTS) / name
    if not path.exists():
        return False
    rows = list(csv.DictReader(open(path)))
    return bool(rows) and int(rows[-1]['round']) >= ROUNDS - 1

runs = list(iter_runs(yaml.safe_load(open(SPEC))))
todo = [(n, c) for n, c in runs if not finished(n)]
print(f'{len(runs) - len(todo)}/{len(runs)} done, {len(todo)} to run')
if not HAS_METRICS:
    print('metrics 모듈이 없어 결과 CSV가 생기지 않는다. 강윤서 님 브랜치가 합쳐진 뒤 실행한다.')
else:
    for i, (name, cmd) in enumerate(todo, 1):
        print(f'[{i}/{len(todo)}] {name}')
        subprocess.run(f'{cmd} trainer.global_rounds={ROUNDS} {COMMON}', shell=True, check=True)